In [1]:
import ee
import time
 
# Initialize Earth Engine
ee.Initialize()
 
# Define Ketapang Regency boundary
ketapang = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(
    ee.Filter.eq('ADM2_NAME', 'Ketapang')
)
 
# Import the BIOPAMA Global Oil Palm dataset
dataset = ee.ImageCollection('BIOPAMA/GlobalOilPalm/v1')
 
# Select the classification band
op_class = dataset.select('classification')
 
# Mosaic all of the granules into a single image
mosaic = op_class.mosaic()
 
# Clip to Ketapang geometry
mosaic_kt = mosaic.clip(ketapang.geometry())

In [2]:
# Create and START export task
task = ee.batch.Export.image.toDrive(
    image=mosaic_kt,
    description='ketapang_biopama_oil_palm',
    folder='GEE_Exports',
    fileNamePrefix='ketapang_biopama_oil_palm',
    region=ketapang.geometry(),
    scale=10,  # Original resolution is ~10m
    maxPixels=1e13,
    crs='EPSG:4326'
)
 
print("Starting export task...")
task.start()
time.sleep(2)  # Give it a moment to register
print("Task status:", task.status())
 
# List all recent tasks to verify
print("\n--- Recent Tasks ---")
tasks = ee.batch.Task.list()
for t in tasks[:5]:
    status = t.status()
    print(f"{status['description']}: {status['state']}")
 
print("\n--- Classification Legend ---")
print("1: Industrial closed-canopy oil palm plantation")
print("2: Smallholder closed-canopy oil palm plantation")
print("3: Other (non-oil palm)")

Starting export task...
Task status: {'state': 'READY', 'description': 'ketapang_biopama_oil_palm', 'priority': 100, 'creation_timestamp_ms': 1759924425189, 'update_timestamp_ms': 1759924425189, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'LOJQR5EM3EP66E4LGKHCXDDH', 'name': 'projects/351618454440/operations/LOJQR5EM3EP66E4LGKHCXDDH'}

--- Recent Tasks ---
ketapang_biopama_oil_palm: READY
ketapang_palm_2023_threshold90: COMPLETED
ketapang_palm_2020_threshold90: COMPLETED
ketapang_palm_2023_threshold70: COMPLETED
ketapang_palm_2020_threshold70: COMPLETED

--- Classification Legend ---
1: Industrial closed-canopy oil palm plantation
2: Smallholder closed-canopy oil palm plantation
3: Other (non-oil palm)
